## data ingestion & cleaning

In [35]:
from datasets import load_dataset, concatenate_datasets

dataset = load_dataset("theatticusproject/cuad-qa", revision="refs/convert/parquet")
train_data = dataset["train"]
test_data = dataset["test"]
print(len(train_data))
print(len(test_data))

22450
4182


In [36]:
full_data = concatenate_datasets([train_data, test_data])

In [37]:
print(len(full_data))

26632


In [38]:
full_data

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 26632
})

In [39]:
seen_titles = set()
unique_contracts = []

for row in full_data:
    if row["title"] not in seen_titles:
        seen_titles.add(row["title"])
        unique_contracts.append({
            "title": row["title"],
            "context": row["context"]
        })
print(f"Number of unique contracts: {len(unique_contracts)}")

Number of unique contracts: 510


In [40]:
import pandas as pd
df = pd.DataFrame(unique_contracts)
print(df["context"].str.len().describe())

count       510.000000
mean      52563.005882
std       55946.484865
min         645.000000
25%       16416.250000
50%       33143.000000
75%       66394.750000
max      338211.000000
Name: context, dtype: float64


In [41]:
print(df["context"].head())

0    EXHIBIT 10.6\n\n                              ...
1    Exhibit 10.26    CONFIDENTIAL TREATMENT HAS BE...
2    Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...
3    REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...
4    Exhibit 10.23 Corporate Address Fannin South P...
Name: context, dtype: object


In [42]:
df.head()

,title,context
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...
1,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...
2,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...
3,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...,REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...
4,"KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSU...",Exhibit 10.23 Corporate Address Fannin South P...


In [43]:
for i in [0, 100]:
    print(f"Contract {i} : {unique_contracts[i]['title']}")
    print(unique_contracts[i]['context'][:800])
    print("\n\n")

Contract 0 : LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its option



Contract 100 : RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEMENT
2008 Sponsorship Agreement- Renewal Sponsor

Exhibit 

In [44]:
eval_examples = [{
    "title" : row["title"],
    "question": row["question"],
    "answers": row["answers"]["text"],
}
for row in full_data]

print(len(eval_examples))
print(eval_examples[0])

26632
{'title': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT', 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract', 'answers': ['DISTRIBUTOR AGREEMENT']}


In [45]:
has_answer = sum(1 for ex in eval_examples if len(ex["answers"]) > 0)
no_answer = len(eval_examples) - has_answer

print(f"with answer: {has_answer} ({has_answer/len(eval_examples) * 100:.1f}%)")
print(f"without answer: {no_answer} ({no_answer/len(eval_examples) * 100:.1f}%)")

with answer: 12424 (46.7%)
without answer: 14208 (53.3%)


In [46]:
categories = set()

for ex in eval_examples:
    q = ex["question"]

    if '"' in q:
        category = q.split('"')[1]
        categories.add(category)

print(f"Number of categories: {len(categories)}")
for c in sorted(categories):
    print(f"- {c}")

Number of categories: 41
- Affiliate License-Licensee
- Affiliate License-Licensor
- Agreement Date
- Anti-Assignment
- Audit Rights
- Cap On Liability
- Change Of Control
- Competitive Restriction Exception
- Covenant Not To Sue
- Document Name
- Effective Date
- Exclusivity
- Expiration Date
- Governing Law
- Insurance
- Ip Ownership Assignment
- Irrevocable Or Perpetual License
- Joint Ip Ownership
- License Grant
- Liquidated Damages
- Minimum Commitment
- Most Favored Nation
- No-Solicit Of Customers
- No-Solicit Of Employees
- Non-Compete
- Non-Disparagement
- Non-Transferable License
- Notice Period To Terminate Renewal
- Parties
- Post-Termination Services
- Price Restrictions
- Renewal Term
- Revenue/Profit Sharing
- Rofr/Rofo/Rofn
- Source Code Escrow
- Termination For Convenience
- Third Party Beneficiary
- Uncapped Liability
- Unlimited/All-You-Can-Eat-License
- Volume Restriction
- Warranty Duration


In [47]:
lengths = df["context"].str.len()
print("smaller contract length:")
print(df.loc[lengths.nsmallest(3).index, ["title"]])
print(f"length: {lengths.nsmallest(3).values}")

print("largest contract length:")
print(df.loc[lengths.nlargest(3).index, ["title"]])
print(f"length: {lengths.nlargest(3).values}")

smaller contract length:
                                                 title
442  RMRGROUPINC_01_22_2020-EX-99.1-JOINT FILING AG...
495  PcquoteComInc_19990721_S-1A_EX-10.11_6377149_E...
501  PRECIGEN,INC_01_22_2020-EX-99.1-JOINT FILING A...
length: [645 857 972]
largest contract length:
                                                 title
358  MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-O...
157  CERES,INC_01_25_2012-EX-10.20-Collaboration Ag...
499  GOOSEHEADINSURANCE,INC_04_02_2018-EX-10.6-Fran...
length: [338211 335282 300768]


In [48]:
import re

sample_text = unique_contracts[0]["context"]

multi_spaces = re.findall(r" {3,}", sample_text)
multi_newlines = re.findall(r"\n{2,}", sample_text)

print(f"Number of multi-spaces (>=3): {len(multi_spaces)}")
print(f"Number of multi-newlines (>=2): {len(multi_newlines)}")
print(f"Text length: {len(sample_text)}")

Number of multi-spaces (>=3): 900
Number of multi-newlines (>=2): 121
Text length: 54290


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

sample_titles = list(lengths.nsmallest(3).index) + list(lengths.nlargest(3).index)

for idx in sample_titles:
    text = df.loc[idx, "context"]
    n_tokens = len(tokenizer.encode(text))
    print(f"{df.loc[idx, 'title'][:50]:<50} | {len(text):>7} chars | {n_tokens:>7} tokens")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (71268 > 512). Running this sequence through the model will result in indexing errors


RMRGROUPINC_01_22_2020-EX-99.1-JOINT FILING AGREEM |     645 chars |     174 tokens
PcquoteComInc_19990721_S-1A_EX-10.11_6377149_EX-10 |     857 chars |     313 tokens
PRECIGEN,INC_01_22_2020-EX-99.1-JOINT FILING AGREE |     972 chars |     263 tokens
MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-OUTSO |  338211 chars |   71268 tokens
CERES,INC_01_25_2012-EX-10.20-Collaboration Agreem |  335282 chars |   78593 tokens
GOOSEHEADINSURANCE,INC_04_02_2018-EX-10.6-Franchis |  300768 chars |   61676 tokens


In [50]:
import re

def clean_contract_text(text: str) -> str:

    text = re.sub(r"[ \t]{2,}", " ", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    text = "\n".join(line.strip() for line in text.split('\n'))

    return text.strip()

sample = unique_contracts[0]["context"]
cleaned = clean_contract_text(sample)

print(f"before cleaning: {len(sample)} chars")
print(f"after cleaning: {len(cleaned)} chars")
print(f"original text:\n{sample[:1000]}\n\n")
print(f"cleaned text:\n{cleaned[:1000]}")

before cleaning: 54290 chars
after cleaning: 38563 chars
original text:
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.

         B. Representations.  As an inducement to the Company to enter into this Agreement,  th

In [51]:
for c in unique_contracts:
    c["clean_context"] = clean_contract_text(c["context"])

reduction = [(len(c["context"]) - len(c["clean_context"])) / len(c["context"]) * 100 for c in unique_contracts]
print(f"average reduction in length after cleaning: {sum(reduction)/len(reduction):.1f}%")

average reduction in length after cleaning: 3.6%


In [52]:
df_cleaned = pd.DataFrame(unique_contracts)

print(df_cleaned["clean_context"].str.len().describe())

count       510.000000
mean      50592.280392
std       53909.954570
min         645.000000
25%       16134.500000
50%       32097.500000
75%       65329.750000
max      300304.000000
Name: clean_context, dtype: float64


## chunking (taille fixe + overlap)

In [54]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Document
from collections import defaultdict

CHUNK_SIZE = 512
CHUNK_OVERLAP = 50

Splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    tokenizer=tokenizer.encode
)

documents = [
    Document(
        text=c["clean_context"], metadata={"title": c["title"]})
        for c in unique_contracts
]

nodes = Splitter.get_nodes_from_documents(documents)

# chunk index for each contract
counters = defaultdict(int)
for n in nodes:
    title = n.metadata["title"]
    n.metadata["chunk_index"] = counters[title]
    counters[title] += 1

print(f"Number of contracts : {len(documents)}")
print(f"Number of chunks: {len(nodes)}")
print(f"Average chunks per contract: {len(nodes)/len(documents):.1f}")

Number of contracts : 510
Number of chunks: 14230
Average chunks per contract: 27.9


In [55]:
chunks_per_contract = pd.Series(counters)
print(chunks_per_contract.describe())

count    510.000000
mean      27.901961
std       30.699734
min        1.000000
25%        9.000000
50%       17.000000
75%       35.000000
max      214.000000
dtype: float64


In [56]:
chunk_token_lengths = [len(tokenizer.encode(n.get_content())) for n in nodes]
pd.Series(chunk_token_lengths).describe()

count    14230.000000
mean       408.228461
std         63.663006
min          3.000000
25%        394.000000
50%        428.000000
75%        447.000000
max        497.000000
dtype: float64

In [57]:
sample_title = unique_contracts[0]["title"]
sample_nodes = [n for n in nodes if n.metadata["title"]==sample_title]

print(f"contract title: {sample_title}")
print(f"total chunks for this contract: {len(sample_nodes)}\n")

for n in sample_nodes[:5]:
    print(f"--- chunk_index={n.metadata['chunk_index']} ({len(tokenizer.encode(n.get_content()))} tokens)")
    print(n.get_content())
    print()

contract title: LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
total chunks for this contract: 18

--- chunk_index=0 (453 tokens)
EXHIBIT 10.6

DISTRIBUTOR AGREEMENT

THIS DISTRIBUTOR AGREEMENT (the "Agreement") is made by and between Electric City Corp., a Delaware corporation ("Company") and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

RECITALS

A. The Company's Business. The Company is presently engaged in the business of selling an energy efficiency device, which is referred to as an "Energy Saver" which may be improved or otherwise changed from its present composition (the "Products"). The Company may engage in the business of selling other products or other devices other than the Products, which will be considered Products if Distributor exercises its options pursuant to Section 7 hereof.

B. Representations. As an inducement to the Company to enter into this Agreement, the Distributor has represented that it has or will have the facilities

In [58]:
contracts_by_title = {c["title"]: c for c in unique_contracts}

single_chunk_titles = [title for title, count in counters.items() if count==1]
print(f"contracts with only 1 chunk : {len(single_chunk_titles)} / {len(unique_contracts)}\n")

for title in single_chunk_titles:
    text = contracts_by_title[title]["clean_context"]
    n_tokens = len(tokenizer.encode(text))
    print(f"{title[:70]:<70} | {len(text):>6} chars | {n_tokens:>4} tokens")

contracts with only 1 chunk : 16 / 510

NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT                       |   1081 chars |  226 tokens
GALACTICOMMTECHNOLOGIESINC_11_07_1997-EX-10.46-WEB HOSTING AGREEMENT   |   1454 chars |  372 tokens
VIRGINGALACTICHOLDINGS,INC_04_08_2020-EX-99.1-JOINT FILING STATEMENT   |   1194 chars |  246 tokens
BANGIINC_05_25_2005-EX-10-Premium Managed Hosting Agreement            |   1973 chars |  446 tokens
BLACKROCKMUNIHOLDINGSINVESTMENTQUALITYFUND_04_07_2020-EX-99.01-JOINT F |   1283 chars |  271 tokens
SPRINGBANKPHARMACEUTICALS,INC_04_08_2020-EX-99.A-JOINT FILING AGREEMEN |   1556 chars |  370 tokens
UNITEDNATIONALBANCORP_03_03_1999-EX-99-Outsourcing Agreement with the  |   1867 chars |  378 tokens
LejuHoldingsLtd_20140121_DRS (on F-1)_EX-10.26_8473102_EX-10.26_Conten |   2112 chars |  451 tokens
MACY_S,INC_05_11_2020-EX-99.4-JOINT FILING AGREEMENT                   |   1397 chars |  329 tokens
GluMobileInc_20070319_S-1A_EX-10.09_436630_EX-10.09_Content 

## Embedding

In [59]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import time

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

sample_chunk = nodes[0].get_content()
start = time.time()
vec = embed_model.get_text_embedding(sample_chunk)
elapsed = time.time() - start

print(f"Embedding dimension: {len(vec)}")
print(f"time for 1 chunk: {elapsed:.3f}s")
print(f"first 5 values: {vec[:5]}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10105.04it/s]


Embedding dimension: 384
time for 1 chunk: 1.108s
first 5 values: [-0.019716884940862656, -0.013978542760014534, 0.03723134845495224, -0.017645804211497307, 0.0216973889619112]


In [60]:
sample_nodes = nodes[:50]

start = time.time()
sample_vecs = embed_model.get_text_embedding_batch([n.get_content() for n in sample_nodes])
elapsed = time.time() - start

print(f"time for {len(sample_nodes)} chunks: {elapsed:.2f}s ({elapsed/len(sample_nodes)*1000:.1f} ms/chunk)")

estimated_total = elapsed/len(sample_nodes)*len(nodes)
print(f"estimated time for all {len(nodes)} chunks: {estimated_total/60:.1f} minutes")

time for 50 chunks: 1.83s (36.5 ms/chunk)
estimated time for all 14230 chunks: 8.7 minutes


## indexation

In [61]:
from qdrant_client import QdrantClient
from llama_index.vector_stores.qdrant import QdrantVectorStore

client = QdrantClient(path="../artifacts/qdrant_storage")
vector_store = QdrantVectorStore(client=client, collection_name="cuad_contracts")

In [62]:
from llama_index.core import VectorStoreIndex, StorageContext

storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(
    nodes,
    embed_model=embed_model,
    storage_context=storage_context,
    show_progress=True,
)

Generating embeddings: 100%|██████████| 2048/2048 [00:25<00:00, 78.85it/s]
/Users/aitbellalhamza/Desktop/RAG-for-contract-understanding-CUAD/.cuad-venv/lib/python3.11/site-packages/llama_index/vector_stores/qdrant/base.py:852: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self._client.create_payload_index(
Generating embeddings: 100%|██████████| 1942/1942 [00:23<00:00, 83.11it/s]


In [64]:
retriever = index.as_retriever(similarity_top_k=3)
results = retriever.retrieve("what is the cap on liability?")

for r in results:
    print(f"score={r.score:.3f} | title={r.metadata['title'][:60]} | chunk_index={r.metadata['chunk_index']}")
    print(r.get_content())
    print()

score=0.751 | title=REGANHOLDINGCORP_03_31_2008-EX-10-LICENSE AND HOSTING AGREEM | chunk_index=14
TAG shall inform LMG of any such Claim against TAG and shall have the right to participate in the defense of any such suit or proceeding at its expense and through counsel of its choosing.

Limitation of Liability. Neither party shall be liable to the other pursuant to this Agreement for any amounts representing loss of profit, loss of business or indirect, consequential, exemplary, or punitive damages of the other party. The foregoing shall not limit the indemnification, defense and hold harmless obligations set forth in this Agreement other than those set forth in Section 5.4 and shall not apply with respect to damages or losses arising from the wrongful termination of this Agreement by LMG, willful misconduct, gross negligence or breach of LMG's obligations under Section 3. 8.

score=0.740 | title=AtnInternationalInc_20191108_10-Q_EX-10.1_11878541_EX-10.1_M | chunk_index=12
2.24 Liabili

In [65]:
results = retriever.retrieve("what is the governing law of this agreement?")

for r in results:
    print(f"score={r.score:.3f} | title={r.metadata['title'][:60]} | chunh_index={r.metadata['chunk_index']}")
    print(r.get_content())
    print()

score=0.763 | title=KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGREEMEN | chunh_index=5
Governing Law. This Agreement shall be governed by and construed in accordance with the laws of the Province of British Columbia (without regard to its conflict of laws provisions) which shall be deemed to be the proper law thereof. All disputes or claims arising out of or in relation to the Agreement may be submitted to and resolved by the Courts of the Province of British Columbia (including the Supreme Court of Canada). The parties hereby irrevocably submit and attorn to the non-exclusive jurisdiction of such Courts to finally adjudicate or determine any suit, action, or proceeding arising out of or in relation to this Agreement.

3

13. Terms Paramount. The terms of this Agreement are in addition to any other existing terms (the "Existing Terms") between the Company and Turpin. In

the event of any inconsistency or conflict between the terms of this Agreement and any Existing Terms, the

## Generation

In [74]:
from llama_index.llms.ollama import Ollama

llm = Ollama(model="llama3.2:3b", request_timeout=120, temperature=0.0)

response = llm.complete("say hello in one sentence.")
print(response)

Hello!


In [76]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine.from_args(retriever=retriever,
                                              llm=llm)

response = query_engine.query("what is the governing law of this agreement?")
print(response)
print("\n--- source chunks used ---")
for node in response.source_nodes:
    print(f"score={node.score:.3f} | title={node.metadata['title']} | chunk_index={node.metadata['chunk_index']}")

The governing law for this agreement is the laws of the Province of British Columbia (without regard to its conflict of law provisions) in some cases, but for other agreements, it is the substantive laws of the State of New York.

--- source chunks used ---
score=0.763 | title=KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGREEMENT | chunk_index=5
score=0.760 | title=VERICELCORP_08_06_2019-EX-10.10-SUPPLY AGREEMENT | chunk_index=50
score=0.760 | title=INTERSECTENT,INC_05_11_2020-EX-10.1-SUPPLY AGREEMENT | chunk_index=17


In [78]:
questions = [
    "Is there a non-compete clause in this agreement?",
    "What happens if either party wants to terminate this agreement for convenience?",
    "What is th cap on liability for damages under this agreement?",
    "Does this agreement give either party audit rights"
]

for q in questions:
    response = query_engine.query(q)
    print(f"QUESTION: {q}")
    print(f"ANSWER: {response}\n")
    print("sources:")
    for node in response.source_nodes:
        print(f" score={node.score:.3f} | title={node.metadata['title']} | chunk_index={node.metadata['chunk_index']}")

    print("\n" + "="*100 + "\n")

QUESTION: Is there a non-compete clause in this agreement?
ANSWER: Yes, there are non-compete clauses in this agreement.

sources:
 score=0.792 | title=Quaker Chemical Corporation - NON COMPETITION AND NON SOLICITATION AGREEMENT | chunk_index=4
 score=0.770 | title=VIVINT SOLAR, INC. - NON-COMPETITION AGREEMENT | chunk_index=1
 score=0.765 | title=VIVINT SOLAR, INC. - NON-COMPETITION AGREEMENT | chunk_index=0


QUESTION: What happens if either party wants to terminate this agreement for convenience?
ANSWER: Neither party can terminate this agreement for convenience.

sources:
 score=0.781 | title=LEJUHOLDINGSLTD_03_12_2014-EX-10.34-INTERNET CHANNEL COOPERATION AGREEMENT | chunk_index=15
 score=0.777 | title=CHEETAHMOBILEINC_04_22_2014-EX-10.43-Cooperation Agreement | chunk_index=11
 score=0.771 | title=TUNIUCORP_03_06_2014-EX-10-COOPERATION AGREEMENT | chunk_index=4


QUESTION: What is th cap on liability for damages under this agreement?
ANSWER: The aggregate of all amounts paid under

In [81]:
sample_title = unique_contracts[107]["title"]

contract_qas = [ex for ex in eval_examples if ex['title'] == sample_title]

print(f"Contract: {sample_title}")
print(f"Number of questions for this contract: {len(contract_qas)}\n")

for qa in contract_qas[:20]:
    print(f"Q: {qa['question']}")
    print(f"A: {qa['answers'] if qa['answers'] else 'no answer/clause absent'}")
    print()

Contract: SLOVAKWIRELESSFINANCECOBV_03_28_2001-EX-4.(B)(II).3-Maintenance and support contract for SICAP(R) modules
Number of questions for this contract: 42

Q: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
A: ['Maintenance and support contract']

Q: Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract
A: ['Sicap Ltd']

Q: Highlight the parts (if any) of this contract related to "Parties" that should be reviewed by a lawyer. Details: The two or more parties who signed the contract
A: ['EuroTel']

Q: Highlight the parts (if any) of this contract related to "Agreement Date" that should be reviewed by a lawyer. Details: The date of the contract
A: no answer/clause absent

Q: Highlight the parts (if any) of this contract related to "Effective Date" that should be reviewed by a lawyer. D

In [86]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter

target_title = "KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGREEMENT"

filtred_retriever = index.as_retriever(
    similarity_top_k=3,
    filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=target_title)]),

)

filtred_query_engine = RetrieverQueryEngine.from_args(retriever=filtred_retriever, llm=llm)

response = filtred_query_engine.query("what is the governing law of this agreement?")
print(response)
print("\nsources:")
for node in response.source_nodes:
    print(f" score={node.score:.3f} | title={node.metadata['title']} | chunk_index={node.metadata['chunk_index']}")

The laws of the Province of British Columbia govern this Agreement.

sources:
 score=0.763 | title=KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGREEMENT | chunk_index=5
 score=0.684 | title=KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGREEMENT | chunk_index=6
 score=0.670 | title=KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGREEMENT | chunk_index=4
